In [0]:
%sql
CREATE OR REPLACE VIEW asb.semantic.data_quality_result AS
SELECT 
    entity_name, 
    total_evaluated, 
    records_passed, 
    records_quarantined,
    run_timestamp
FROM asb.quarantine.audit_pipeline_runs
;

In [0]:
%sql
-- Unified metrics across all quarantined entities
CREATE OR REPLACE VIEW asb.semantic.data_quality_metrics AS
SELECT 
    entity_name,
    record_key,
    _failure_count,
    failed_rule,
    SPLIT(failed_rule, ':')[0] AS rule_id,
    TRIM(SPLIT(failed_rule, ':')[1]) AS failure_reason,
    _quarantined_at
FROM (
    SELECT 
        entity_name, 
        record_key, 
        _failure_count, 
        EXPLODE(failed_rules) AS failed_rule, 
        _quarantined_at 
    FROM (
        SELECT entity_name, record_key, _failure_count, failed_rules, _quarantined_at FROM asb.quarantine.quarantine_orders
        UNION ALL
        SELECT entity_name, record_key, _failure_count, failed_rules, _quarantined_at FROM asb.quarantine.quarantine_order_items
        UNION ALL
        SELECT entity_name, record_key, _failure_count, failed_rules, _quarantined_at FROM asb.quarantine.quarantine_order_payments
        UNION ALL
        SELECT entity_name, record_key, _failure_count, failed_rules, _quarantined_at FROM asb.quarantine.quarantine_order_reviews
        UNION ALL
        SELECT entity_name, record_key, _failure_count, failed_rules, _quarantined_at FROM asb.quarantine.quarantine_products
        UNION ALL
        SELECT entity_name, record_key, _failure_count, failed_rules, _quarantined_at FROM asb.quarantine.quarantine_customers
        UNION ALL
        SELECT entity_name, record_key, _failure_count, failed_rules, _quarantined_at FROM asb.quarantine.quarantine_sellers
        UNION ALL
        SELECT entity_name, record_key, _failure_count, failed_rules, _quarantined_at FROM asb.quarantine.quarantine_geolocation
    )
)
;